In [1]:
import os
os.chdir('../../../')

In [2]:
!ls

backbones  example  README.md	      runs     submodules
command    LICENSE  reports	      sh       training
datasets   prompts  requirements.txt  solvers  utils


In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import numpy as np
import time
import torch
from utils.general_classifier import Classifier   # <- 네가 정의해둔 Classifier 모듈/경로에 맞게 수정

# (arch, weight_fullname) — 문자열 그대로 Classifier에 넘겨 사용
MODELS_FOR_EXP = [
  ("shufflenet_v2_x0_5", "ShuffleNet_V2_X0_5_Weights.IMAGENET1K_V1"),
  ("mobilenet_v3_small", "MobileNet_V3_Small_Weights.IMAGENET1K_V1"),
  ("mnasnet0_75",        "MNASNet0_75_Weights.IMAGENET1K_V1"),
  ("mobilenet_v2",       "MobileNet_V2_Weights.IMAGENET1K_V1"),
  ("efficientnet_b0",    "EfficientNet_B0_Weights.IMAGENET1K_V1"),
  ("regnet_y_400mf",     "RegNet_Y_400MF_Weights.IMAGENET1K_V1"),
  ("alexnet",            "AlexNet_Weights.IMAGENET1K_V1"),
  ("regnet_x_800mf",     "RegNet_X_800MF_Weights.IMAGENET1K_V1"),
  ("efficientnet_b2",    "EfficientNet_B2_Weights.IMAGENET1K_V1"),
  ("densenet121",        "DenseNet121_Weights.IMAGENET1K_V1"),
  ("resnet50",           "ResNet50_Weights.IMAGENET1K_V1"),
  ("vit_b_32",           "ViT_B_32_Weights.IMAGENET1K_V1"),
  ("swin_t",             "Swin_T_Weights.IMAGENET1K_V1"),
  ("maxvit_t",           "MaxVit_T_Weights.IMAGENET1K_V1"),
  ("inception_v3",       "Inception_V3_Weights.IMAGENET1K_V1"),  # 포함 요구
  ("convnext_small",     "ConvNeXt_Small_Weights.IMAGENET1K_V1"),
  ("vit_b_16",           "ViT_B_16_Weights.IMAGENET1K_V1"),      # 포함 요구
  ("efficientnet_v2_m",  "EfficientNet_V2_M_Weights.IMAGENET1K_V1"),
  ("regnet_y_32gf",      "RegNet_Y_32GF_Weights.IMAGENET1K_V1"),
  ("vit_l_16",           "ViT_L_16_Weights.IMAGENET1K_V1"),
]


DEVICE = "cpu"   # transforms 호환성 위해 CPU 권장; GPU 쓰려면 "cuda"로 바꾸고 Classifier가 GPU에서도 잘 동작하는지 확인
BATCH_SIZE = 1
H0, W0 = 256, 256   # 더미 입력 크기(가중치 transforms가 내부에서 정해진 크기로 리사이즈함)

def make_dummy(batch=BATCH_SIZE, h=H0, w=W0, device=DEVICE):
    # Classifier 기본 input_range="-1..1" 이므로 그 범위로 생성
    return (torch.rand(batch, 3, h, w, device=device) * 2.0) - 1.0

@torch.no_grad()
def run_one(arch: str, weight_fullname: str):
    print(f"\n[{arch} | {weight_fullname}]")
    try:
        model = Classifier(
            arch=arch,
            weights=weight_fullname
        )
    except Exception as e:
        print(f"  ✗ Classifier 초기화 실패: {e}")
        return

    x = make_dummy()
    try:
        dts = []
        for _ in range(10):
            t0 = time.time()
            out = model(x)
            dt = (time.time() - t0) * 1000.0
            dts.append(dt)
        dt = np.mean(dts)
        logits = out["logits"]
        assert isinstance(logits, torch.Tensor), "logits가 Tensor가 아님"
        assert logits.ndim == 2 and logits.shape[0] == BATCH_SIZE, f"출력 shape 이상: {tuple(logits.shape)}"
        print(f"  ✓ forward OK — logits: {tuple(logits.shape)} — {dt:.1f} ms")
    except Exception as e:
        print(f"  ✗ forward 실패: {e}")
    finally:
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

def main():
    print(f"Device: {DEVICE}, torch {torch.__version__}")
    for arch, w in MODELS_FOR_EXP:
        run_one(arch, w)
    print("\nDone.")

if __name__ == "__main__":
    main()


Device: cpu, torch 2.8.0+cu128

[shufflenet_v2_x0_5 | ShuffleNet_V2_X0_5_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 9.7 ms

[mobilenet_v3_small | MobileNet_V3_Small_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 4.3 ms

[mnasnet0_75 | MNASNet0_75_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 5.7 ms

[mobilenet_v2 | MobileNet_V2_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 5.8 ms

[efficientnet_b0 | EfficientNet_B0_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 10.0 ms

[regnet_y_400mf | RegNet_Y_400MF_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 26.1 ms

[alexnet | AlexNet_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 3.5 ms

[regnet_x_800mf | RegNet_X_800MF_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 6.2 ms

[efficientnet_b2 | EfficientNet_B2_Weights.IMAGENET1K_V1]
  ✓ forward OK — logits: (1, 1000) — 14.7 ms

[densenet121 | DenseNet121_Weights.IMAGENET1K_V1]
  ✓ forwar